# Reasoning Models

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/07-reasoning-models)

This notebook explores test-time compute scaling: simulating how accuracy improves with more thinking tokens, implementing the self-consistency majority-vote method, and visualizing the train-time vs. test-time scaling trade-offs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(0)

## 1. Test-time compute: more tokens → higher accuracy

We simulate a reasoning model whose per-step error probability decreases as it is given more 'thinking' tokens (more steps to self-correct).

In [ ]:
def simulate_accuracy(n_thinking_tokens, base_accuracy=0.40, n_problems=2000, rng=None):
    """
    Simulate reasoning model accuracy as a function of thinking token budget.
    Each additional 'block' of 256 tokens lets the model catch errors with some probability.
    """
    rng = rng or np.random.RandomState(42)
    blocks = max(1, n_thinking_tokens // 256)
    # Each block has an independent chance of being the one that finds the solution
    acc = 1 - (1 - base_accuracy) ** blocks  # coverage with diminishing returns
    # Add noise for realism
    return np.clip(acc + rng.randn() * 0.01, 0, 1)

token_budgets = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
accs_easy   = [simulate_accuracy(t, base_accuracy=0.75) for t in token_budgets]
accs_medium = [simulate_accuracy(t, base_accuracy=0.45) for t in token_budgets]
accs_hard   = [simulate_accuracy(t, base_accuracy=0.20) for t in token_budgets]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(token_budgets, accs_easy,   'o-', color='#10b981', linewidth=2, label='Easy problems')
ax.semilogx(token_budgets, accs_medium, 's-', color='#f59e0b', linewidth=2, label='Medium problems')
ax.semilogx(token_budgets, accs_hard,   '^-', color='#f43f5e', linewidth=2, label='Hard problems')
ax.set_xlabel('Thinking token budget (log scale)')
ax.set_ylabel('Simulated accuracy')
ax.set_title('Test-Time Compute Scaling: Accuracy vs. Thinking Tokens', color='#e2e8f0')
ax.legend()
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

## 2. Self-consistency: majority vote over reasoning chains

Self-consistency samples k independent reasoning chains and takes the majority vote. Each chain gets the correct answer with probability p; voting improves robustness.

In [ ]:
def self_consistency_accuracy(p_correct_per_chain, k_samples, n_trials=10000, seed=0):
    """
    Probability that majority vote is correct when each chain is correct with prob p.
    """
    rng = np.random.RandomState(seed)
    votes = rng.rand(n_trials, k_samples) < p_correct_per_chain  # True = correct answer
    majority_correct = votes.sum(axis=1) > k_samples / 2
    return majority_correct.mean()

k_values = [1, 3, 5, 9, 15, 25]
p_values = [0.40, 0.55, 0.70]
colors   = ['#f43f5e', '#f59e0b', '#10b981']

fig, ax = plt.subplots(figsize=(9, 5))
for p, color in zip(p_values, colors):
    accs = [self_consistency_accuracy(p, k) for k in k_values]
    ax.plot(k_values, accs, 'o-', color=color, linewidth=2, label=f'Single-chain acc = {int(p*100)}%')
    ax.axhline(p, color=color, linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('Number of sampled chains (k)')
ax.set_ylabel('Majority-vote accuracy')
ax.set_title('Self-Consistency: How Many Chains Do You Need?', color='#e2e8f0')
ax.legend()
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

print("Key insight: self-consistency helps most when each chain is moderately reliable (50-70%).")
print("With only 40% per chain, even 25 samples may not rescue accuracy.")

## 3. Train-time vs. test-time scaling curves

In [ ]:
# Simulated: train-time RL steps improve base accuracy (shifts the curve up)
# Test-time token budget then further improves accuracy given the base

def accuracy_combined(rl_steps, token_budget, base=0.15):
    # More RL steps raise the base accuracy
    trained_base = base + (1 - base) * (1 - np.exp(-rl_steps / 5000))
    # More test-time tokens improve from the trained base
    blocks = max(1, token_budget // 256)
    return 1 - (1 - trained_base) ** blocks

rl_steps_list = [0, 1000, 3000, 8000, 20000]
token_budgets_cont = np.logspace(np.log10(64), np.log10(8192), 40)
colors_rl = ['#4b5563', '#6366f1', '#22d3ee', '#f59e0b', '#10b981']

fig, ax = plt.subplots(figsize=(10, 5))
for rl, color in zip(rl_steps_list, colors_rl):
    accs = [accuracy_combined(rl, t) for t in token_budgets_cont]
    label = f'RL steps = {rl:,}' if rl > 0 else 'No RL training'
    ax.semilogx(token_budgets_cont, accs, color=color, linewidth=2, label=label)

ax.set_xlabel('Test-time token budget (log scale)')
ax.set_ylabel('Accuracy')
ax.set_title('Combined Train-Time + Test-Time Compute Scaling', color='#e2e8f0')
ax.legend(title='Train-time RL', title_fontsize=9)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

print("Both levers help and they compose: more RL training raises the ceiling that test-time tokens can reach.")

## ✏️ Your turn

**Exercise 1 – Best-of-N sampling.** Best-of-N (BoN) generates N candidates and picks the best using a reward model. Simulate BoN: each candidate is correct with probability p; the final answer is correct if ANY candidate is correct. Plot BoN accuracy vs. N for p ∈ {0.3, 0.5, 0.7}. How does BoN compare to self-consistency for the same N?

In [ ]:
N_values = [1, 2, 4, 8, 16, 32]
p_values = [0.30, 0.50, 0.70]

# TODO(you): for each p and N, compute P(at least one of N is correct) = 1 - (1-p)^N
# Then plot and compare to self_consistency_accuracy(p, N) from above

In [ ]:
# Assert cell
for p in [0.30, 0.50, 0.70]:
    bon_32 = 1 - (1 - p) ** 32
    sc_25  = self_consistency_accuracy(p, 25)
    print(f"p={p:.0%}: BoN@32={bon_32:.3f}, Self-cons@25={sc_25:.3f}")
    # BoN@32 should always be ≥ p
    assert bon_32 >= p - 0.01

<details><summary>Solution</summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
for p, color in zip(p_values, ['#f43f5e', '#f59e0b', '#10b981']):
    bon_accs = [1 - (1 - p) ** n for n in N_values]
    sc_accs  = [self_consistency_accuracy(p, n) for n in N_values]
    ax.plot(N_values, bon_accs, 'o-', color=color, linewidth=2, label=f'BoN p={p:.0%}')
    ax.plot(N_values, sc_accs,  's--', color=color, linewidth=1.5, alpha=0.7, label=f'SC p={p:.0%}')
ax.set_xlabel('N (candidates / samples)')
ax.set_ylabel('Accuracy')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()
```

BoN needs a reliable reward model (otherwise you pick a wrong answer confidently). Self-consistency only requires the majority to agree, which is robust to reward model errors but needs the correct answer to be the most common one.

</details>